# GUI for CatBoost Downstream Reference Prediction


In [ ]:
from pathlib import Path
import tkinter as tk
from tkinter import messagebox

import pandas as pd
from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split

ROOT = Path.cwd()
DATA_PATH = ROOT / "frc_compressive_strength.csv"
TARGET = "CS"
RANDOM_STATE = 42
FEATURES = [
    "C", "SF", "FA", "Fine_Agg", "Coarse_Agg", "W", "SP", "D",
    "W_C_ratio", "Total_Binder", "Total_Fiber", "SF_C_ratio",
    "W_Binder_ratio", "Fiber_Aspect_Ratio",
]

df = pd.read_csv(DATA_PATH)
x_train, _, y_train, _ = train_test_split(df[FEATURES], df[TARGET], test_size=0.2, random_state=RANDOM_STATE)
model = CatBoostRegressor(iterations=1200, depth=4, learning_rate=0.1, l2_leaf_reg=3.0, loss_function="RMSE", random_seed=RANDOM_STATE, verbose=False)
model.fit(x_train, y_train)

app = tk.Tk()
app.title("FRC Compressive Strength Predictor")
entries = {}
for row, feature in enumerate(FEATURES):
    tk.Label(app, text=feature, width=22, anchor="w").grid(row=row, column=0, padx=8, pady=3)
    entry = tk.Entry(app, width=16)
    entry.grid(row=row, column=1, padx=8, pady=3)
    entries[feature] = entry

result = tk.Label(app, text="Predicted CS: -- MPa", font=("Arial", 12, "bold"))

def predict():
    try:
        values = [float(entries[feature].get()) for feature in FEATURES]
    except ValueError:
        messagebox.showerror("Input error", "Please enter numeric values for all features.")
        return
    prediction = model.predict(pd.DataFrame([values], columns=FEATURES))[0]
    result.configure(text=f"Predicted CS: {prediction:.2f} MPa")

tk.Button(app, text="Predict", command=predict).grid(row=len(FEATURES), column=0, columnspan=2, padx=8, pady=10, sticky="ew")
result.grid(row=len(FEATURES) + 1, column=0, columnspan=2, padx=8, pady=8)
app.mainloop()
